# WAV 轉 16 kHz Notebook

這個 Notebook 可以把輸入的 `.wav` 轉成 **16 kHz**，支援：

- 單一 WAV 檔轉換
- 整個資料夾批次轉換
- 保留單聲道或多聲道格式
- 輸出為 16-bit PCM WAV


## 1. 安裝套件

如果你的環境已經有 `soundfile` 和 `scipy`，這格可以跳過。

In [ ]:
%pip install soundfile scipy

## 2. 匯入套件與定義轉換函式

In [ ]:
from pathlib import Path
from math import gcd

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly


def convert_wav_to_16k(input_wav, output_wav=None, target_sr=16000, subtype="PCM_16"):
    """
    Convert a WAV file to 16 kHz.

    Args:
        input_wav (str or Path): Input WAV path.
        output_wav (str or Path, optional): Output WAV path.
            If None, output will be saved as <original_name>_16k.wav.
        target_sr (int): Target sampling rate. Default: 16000.
        subtype (str): WAV subtype. Default: PCM_16.

    Returns:
        Path: Output WAV path.
    """
    input_wav = Path(input_wav)

    if not input_wav.exists():
        raise FileNotFoundError(f"Input file not found: {input_wav}")

    if output_wav is None:
        output_wav = input_wav.with_name(input_wav.stem + "_16k.wav")
    else:
        output_wav = Path(output_wav)

    output_wav.parent.mkdir(parents=True, exist_ok=True)

    # always_2d=True keeps shape as (samples, channels)
    audio, sr = sf.read(input_wav, dtype="float32", always_2d=True)

    if sr != target_sr:
        factor = gcd(sr, target_sr)
        up = target_sr // factor
        down = sr // factor
        audio = resample_poly(audio, up=up, down=down, axis=0)

    # Avoid clipping when saving PCM_16
    audio = np.clip(audio, -1.0, 1.0)

    # If original is mono, save as mono instead of (N, 1)
    if audio.shape[1] == 1:
        audio = audio[:, 0]

    sf.write(output_wav, audio, target_sr, subtype=subtype)

    print(f"Input : {input_wav}")
    print(f"Output: {output_wav}")
    print(f"Original sample rate: {sr} Hz")
    print(f"Target sample rate  : {target_sr} Hz")

    return output_wav

## 3. 單一 WAV 檔轉成 16 kHz

把 `input_wav` 改成你的檔案路徑即可。

In [ ]:
input_wav = "input.wav"  # 改成你的 wav 檔，例如 "./audio/test.wav"

output_wav = convert_wav_to_16k(input_wav)
output_wav

## 4. 指定輸出檔名

也可以自己指定輸出路徑。

In [ ]:
input_wav = "input.wav"
output_wav = "output_16k.wav"

convert_wav_to_16k(input_wav, output_wav)

## 5. 批次轉換整個資料夾

把 `input_dir` 改成放 WAV 的資料夾。轉好的檔案會放到 `output_dir`。

In [ ]:
input_dir = Path("wav_in")      # 放原始 wav 的資料夾
output_dir = Path("wav_16k")    # 輸出 16k wav 的資料夾

output_dir.mkdir(parents=True, exist_ok=True)

wav_files = sorted(input_dir.glob("*.wav"))
print(f"Found {len(wav_files)} wav files.")

for wav_path in wav_files:
    out_path = output_dir / wav_path.name
    convert_wav_to_16k(wav_path, out_path)
    print("-" * 60)

## 6. 檢查輸出 WAV 資訊

In [ ]:
# 檢查單一檔案
sf.info(output_wav)